In [ ]:

import pandas as pd
%pip install openpyxl
file_path = r'C:\AjayDAREPO\python\projectlloyds\Customer_Churn_Data_Large (1).xlsx'
# 1. EXACT filenames from your sidebar
file_demo = 'Customer_Churn_Data_Large (1).xlsx - Customer_Demographics.csv'
file_online = 'Customer_Churn_Data_Large (1).xlsx - Online_Activity.csv'
file_trans = 'Customer_Churn_Data_Large (1).xlsx - Transaction_History.csv'
file_service = 'Customer_Churn_Data_Large (1).xlsx - Customer_Service.csv'
file_churn = 'Customer_Churn_Data_Large (1).xlsx - Churn_Status.csv'

# 2. Load each sheet into a separate Dataframe
df_churn = pd.read_excel(file_path, sheet_name='Churn_Status')
df_demo = pd.read_excel(file_path, sheet_name='Customer_Demographics')
df_service = pd.read_excel(file_path, sheet_name='Customer_Service')
df_online = pd.read_excel(file_path, sheet_name='Online_Activity')
df_trans = pd.read_excel(file_path, sheet_name='Transaction_History')


print("Success! All sheets loaded from the XLSX file.")


 # 3. Aggregate 
df_trans_agg = df_trans.groupby('CustomerID').agg({
    'AmountSpent': 'sum',
    'TransactionID': 'count'
}).rename(columns={'AmountSpent': 'TotalSpent', 'TransactionID': 'Txn_Count'}).reset_index()

df_service_agg = df_service.groupby('CustomerID').agg({
    'InteractionID': 'count'
}).rename(columns={'InteractionID': 'Service_Interactions'}).reset_index()

# 4. Master Merge
master_df = df_demo.merge(df_online, on='CustomerID', how='left') \
                   .merge(df_trans_agg, on='CustomerID', how='left') \
                   .merge(df_service_agg, on='CustomerID', how='left') \
                   .merge(df_churn, on='CustomerID', how='left')

# 5. Fill NaNs with 0
master_df[['TotalSpent', 'Txn_Count', 'Service_Interactions']] = master_df[['TotalSpent', 'Txn_Count', 'Service_Interactions']].fillna(0)

print("Master Table Created! Here is the head:")
master_df.info()
# 1. Check the Churn balance
print(master_df['ChurnStatus'].value_counts())

# 2. Check for any missing values after the merge
print(master_df.isnull().sum())
# Save the final merged data to a new CSV file
master_df.to_csv('Lloyds_Merged_Data_Final.csv', index=False)

# OR save it as an Excel file
master_df.to_excel('Lloyds_Merged_Data_Final.xlsx', index=False)

print("The new merged file is ready.")

# 1. See how many people Churned (1) vs Stayed (0)
print("--- Churn Count ---")
print(master_df['ChurnStatus'].value_counts())


# 2. See the Average Age and Spending of Churners
print("\n--- Averages by Churn Group ---")
print(master_df.groupby('ChurnStatus')[['Age', 'TotalSpent']].mean())


# Check Churn Percentage
print("Churn Rate:", master_df['ChurnStatus'].mean())

In [ ]:

# Filter for only those who left (ChurnStatus == 1)
churned_only = master_df[master_df['ChurnStatus'] == 1]

print("--- Profile of the 204 People who Left ---")
print(churned_only[['Age', 'TotalSpent', 'Service_Interactions']].describe())

# Check their most common gender or income level
print("\nMost common Income Level for Churners:")
print(churned_only['IncomeLevel'].value_counts())


In [ ]:
print(master_df.columns)

In [ ]:




# 1. Compare the centers using the ACTUAL columns
# Notice: No underscore in LoginFrequency
behavior_stats = master_df.groupby('ChurnStatus')[['Age', 'TotalSpent', 'LoginFrequency', 'Service_Interactions']].agg(['mean', 'std', 'median'])

print("--- Behavioral Comparison (0: Stayers vs 1: Churners) ---")
print(behavior_stats)

# 2. Percentage Difference Check
# This will show the % gap for LoginFrequency specifically
means = master_df.groupby('ChurnStatus').mean(numeric_only=True)
pct_diff = ((means.loc[1] - means.loc[0]) / means.loc[0] * 100)

print("\n--- Feature Impact Score (% Difference) ---")
print(pct_diff.sort_values(ascending=False))

# 3. Visualizing the Login Gap
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.kdeplot(data=master_df, x='LoginFrequency', hue='ChurnStatus', fill=True, palette='crest')
plt.title('The "Silent" Churn Signal: Login Frequency')
plt.show()

In [ ]:
# 1. Create a copy for deep-diving into categories
hidden_hints_df = master_df.copy()

# 2. Convert all text categories into numbers (One-Hot Encoding)
# We focus on Gender, MaritalStatus, and IncomeLevel
encoded_df = pd.get_dummies(hidden_hints_df, columns=['Gender', 'MaritalStatus', 'IncomeLevel'], drop_first=True)

# 3. Now run the correlation again including these new numbers
new_corr = encoded_df.select_dtypes(include=['number']).corr()

print("--- NEW Hidden Hints Correlation (Top Drivers) ---")
# Look for the strongest relationships with ChurnStatus
print(new_corr['ChurnStatus'].sort_values(ascending=False))

In [ ]:
# Check Churn Rate by Marital Status
marital_check = master_df.groupby('MaritalStatus')['ChurnStatus'].mean() * 100
print("--- Churn Rate % by Marital Status ---")
print(marital_check)

# Check Churn Rate by Gender
gender_check = master_df.groupby('Gender')['ChurnStatus'].mean() * 100
print("\n--- Churn Rate % by Gender ---")
print(gender_check)

# Check Churn Rate by Income Level
income_check = master_df.groupby('IncomeLevel')['ChurnStatus'].mean() * 100
print("\n--- Churn Rate % by Income Level ---")
print(income_check)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. HISTOGRAM: Age Distribution
sns.histplot(data=master_df, x='Age', hue='ChurnStatus', kde=True, bins=30, palette='magma', ax=axes[0,0])
axes[0,0].set_title('1. Age Distribution: Identifying the Churn Cluster')
# Moving legend outside
sns.move_legend(axes[0,0], "upper left", bbox_to_anchor=(1, 1))

# 2. SCATTER PLOT: Income vs Total Spent
sns.scatterplot(data=master_df, x='IncomeLevel', y='TotalSpent', hue='ChurnStatus', alpha=0.6, ax=axes[0,1])
axes[0,1].set_title('2. Income vs Spent: High-Value Churn Assessment')
# Moving legend outside
axes[0,1].legend(title='ChurnStatus', bbox_to_anchor=(1.05, 1), loc='upper left')

# 3. BOX PLOT: Login Frequency
sns.boxplot(data=master_df, x='ChurnStatus', y='LoginFrequency', palette='coolwarm', ax=axes[1,0])
axes[1,0].set_title('3. Login Frequency: The Primary Behavioral Driver')

# 4. CORRELATION MATRIX: Heatmap (Heatmaps usually don't have overlapping legends)
numeric_cols = ['Age', 'TotalSpent', 'LoginFrequency', 'Service_Interactions', 'ChurnStatus']
corr_matrix = master_df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, ax=axes[1,1])
axes[1,1].set_title('4. Feature Correlation Matrix (Numeric)')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. HANDLE MISSING VALUES
# Context: In bank data, missing activity usually means 'Zero' activity, not 'Unknown' data.
master_df[['TotalSpent', 'Service_Interactions', 'LoginFrequency']] = master_df[['TotalSpent', 'Service_Interactions', 'LoginFrequency']].fillna(0)

# 2. ADDRESS OUTLIERS
# Context: We 'Cap' outliers at the 99th percentile so extreme 'Whales' don't confuse the model.
q_limit = master_df['TotalSpent'].quantile(0.99)
master_df['TotalSpent'] = master_df['TotalSpent'].clip(upper=q_limit)

# 3. ENCODE CATEGORICAL VARIABLES (One-Hot Encoding)
# Turning text into numbers so the ML model can 'read' them.
df_preprocessed = pd.get_dummies(master_df, columns=['Gender', 'MaritalStatus', 'IncomeLevel'], drop_first=True)

# 4. STANDARDIZE NUMERICAL FEATURES
# Crucial for models like Logistic Regression or SVM so 'TotalSpent' (1000s) doesn't overpower 'Age' (10s).
scaler = StandardScaler()
num_cols = ['Age', 'TotalSpent', 'LoginFrequency']
df_preprocessed[num_cols] = scaler.fit_transform(df_preprocessed[num_cols])

print("--- Preprocessing Complete ---")
print(f"Final Data Shape: {df_preprocessed.shape}")
df_preprocessed.head()



In [ ]:
# List of columns to drop
# 1. CustomerID: Irrelevant unique identifier
# 2. LastLoginDate: Redundant (already represented by LoginFrequency)
# 3. ServiceUsage: Categorical noise not included in our Rationale
# 4. Txn_Count: Redundant (High correlation with TotalSpent)

cols_to_drop = ['CustomerID', 'LastLoginDate', 'ServiceUsage', 'Txn_Count']
# Dropping the columns
df_final = df_preprocessed.drop(columns=cols_to_drop)

print("--- Final Clean-Up Complete ---")
print(f"Remaining Columns: {df_final.columns.tolist()}")
print(f"Final shape for Model Building: {df_final.shape}")

# Show the clean, purely numeric version
df_final.head()

In [ ]:
df_final.to_csv('Lloyds_Ready_For_ML.csv', index=False)

In [ ]:
!pip install imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE

In [ ]:
np.random.seed(42)

In [ ]:
# ==========================================
# PHASE A: DATA PREPARATION
# ==========================================

X = df_final.drop(columns=['ChurnStatus'])
y = df_final['ChurnStatus']

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

print("\nOriginal Class Distribution:")
print(y_train.value_counts())

In [ ]:
y.value_counts()

In [ ]:
# ==========================================
# PHASE B: APPLY SMOTE
# ==========================================

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

In [ ]:
# ==========================================
# PHASE C: STRATIFIED CROSS VALIDATION
# ==========================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
# ==========================================
# PHASE D: MODEL SETUP
# ==========================================

base_model = RandomForestClassifier(
    random_state=42
)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5]
}

In [ ]:
# ==========================================
# PHASE E: GRID SEARCH
# ==========================================

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=skf,
    scoring='balanced_accuracy',
    n_jobs=-1
)

# Train ONLY on SMOTE training data
grid_search.fit(X_train_smote, y_train_smote)

print("Best Parameters:")
print(grid_search.best_params_)

print(f"\nBest CV Balanced Accuracy: {grid_search.best_score_ * 100:.2f}%")

# Best tuned model
tuned_model = grid_search.best_estimator_

In [ ]:
# ==========================================
# PHASE F: PREDICTIONS
# ==========================================

# Predict probabilities on untouched test set
y_probs = tuned_model.predict_proba(X_test)[:, 1]

# Default threshold
threshold = 0.50

y_pred = (y_probs >= threshold).astype(int)

print(f"Threshold Used: {threshold}")

In [ ]:
# ==========================================
# PHASE G: EVALUATION
# ==========================================

balanced_acc = balanced_accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_probs)

print(f"Balanced Accuracy: {balanced_acc * 100:.2f}%")
print(f"ROC-AUC Score: {roc_auc:.3f}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

In [ ]:
# ==========================================
# PHASE H: CONFUSION MATRIX
# ==========================================

plt.figure(figsize=(6, 4))

sns.heatmap(
    confusion_matrix(y_test, y_pred),
    annot=True,
    fmt='d',
    cmap='Greens'
)

plt.title('Confusion Matrix - SMOTE Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()

In [ ]:
# ==========================================
# PHASE I: FEATURE IMPORTANCE
# ==========================================

importances = pd.Series(
    tuned_model.feature_importances_,
    index=X.columns
)

plt.figure(figsize=(10, 6))

importances.sort_values().plot(
    kind='barh',
    color='teal'
)

plt.title('Feature Importance: Drivers of Customer Churn')
plt.xlabel('Importance Score')

plt.show()

**"An alternative gradient boosting approach (XGBoost) was explored to improve minority-class detection."**

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
# ==========================================
# XGBOOST MODEL
# ==========================================

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0]
}

In [ ]:
# ==========================================
# XGBOOST GRID SEARCH
# ==========================================

grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    cv=skf,
    scoring='balanced_accuracy',
    n_jobs=-1
)

# Train on SMOTE training data
grid_search_xgb.fit(X_train_smote, y_train_smote)

print("Best Parameters:")
print(grid_search_xgb.best_params_)

print(f"\nBest CV Balanced Accuracy: {grid_search_xgb.best_score_ * 100:.2f}%")

best_xgb = grid_search_xgb.best_estimator_

In [ ]:
# ==========================================
# XGBOOST PREDICTIONS
# ==========================================

y_probs_xgb = best_xgb.predict_proba(X_test)[:, 1]

threshold = 0.3

y_pred_xgb = (y_probs_xgb >= threshold).astype(int)

In [ ]:
# ==========================================
# XGBOOST EVALUATION
# ==========================================

balanced_acc_xgb = balanced_accuracy_score(y_test, y_pred_xgb)

roc_auc_xgb = roc_auc_score(y_test, y_probs_xgb)

print(f"Balanced Accuracy: {balanced_acc_xgb * 100:.2f}%")
print(f"ROC-AUC Score: {roc_auc_xgb:.3f}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
# ==========================================
# XGBOOST CONFUSION MATRIX
# ==========================================

plt.figure(figsize=(6, 4))

sns.heatmap(
    confusion_matrix(y_test, y_pred_xgb),
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('Confusion Matrix - XGBoost')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()

In [ ]:

# Feature Importance
importances = pd.Series(
    best_xgb.feature_importances_,
    index=X.columns
)

plt.figure(figsize=(10,6))
importances.sort_values().plot(kind='barh')

plt.title('XGBoost Feature Importance')
plt.xlabel('Importance Score')

plt.show()

print(importances.sort_values(ascending=False))

In [ ]:
# ==========================================
# MLFLOW LOGGING - XGBOOST
# ==========================================

import mlflow
import mlflow.xgboost
from sklearn.metrics import f1_score, precision_score, recall_score
import os

# Force artifacts to go THROUGH the server, not directly to filesystem
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_ARTIFACT_URI"] = "http://localhost:5000/api/2.0/mlflow-artifacts/"

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Lloyds_Churn_Production_Project")
with mlflow.start_run(run_name="Final_XGBoost_SMOTE"):

    # ---------- Params ----------
    mlflow.log_params(grid_search_xgb.best_params_)
    mlflow.log_param("threshold", threshold)
    mlflow.log_param("cv_strategy", "StratifiedKFold")
    mlflow.log_param("scoring", "balanced_accuracy")
    mlflow.log_param("smote_applied", True)

    # ---------- Metrics ----------
    mlflow.log_metric("balanced_accuracy", balanced_acc_xgb)
    mlflow.log_metric("roc_auc", roc_auc_xgb)
    mlflow.log_metric("best_cv_balanced_accuracy", grid_search_xgb.best_score_)

    mlflow.log_metric("f1_churn",        f1_score(y_test, y_pred_xgb, pos_label=1))
    mlflow.log_metric("f1_no_churn",     f1_score(y_test, y_pred_xgb, pos_label=0))
    mlflow.log_metric("precision_churn", precision_score(y_test, y_pred_xgb, pos_label=1))
    mlflow.log_metric("recall_churn",    recall_score(y_test, y_pred_xgb, pos_label=1))

    # ---------- Model ----------
    mlflow.xgboost.log_model(
        xgb_model=best_xgb,
        artifact_path="model",
        registered_model_name="Lloyds_XGBoost_Churn_Model"
    )

    print("MLflow logging successful!")


# ==========================================
# SAVE PICKLE
# ==========================================

import pickle

with open("best_xgb_model.pkl", "wb") as f:
    pickle.dump(best_xgb, f)

print("Pickle saved!")

In [82]:
import os
print(os.getcwd())
print(os.path.exists("best_xgb_model.pkl"))


c:\AjayDAREPO\python\projectlloyds
True


In [ ]:
import mlflow
import os

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Lloyds_Churn_Production_Project")

with mlflow.start_run(run_name="XGBoost_Final") as run:
    mlflow.log_artifact("best_xgb_model.pkl", artifact_path="model")
    print("Run ID:", run.info.run_id)
    print("Done!")

In [83]:
import mlflow

client = mlflow.tracking.MlflowClient("http://localhost:5000")
runs = client.search_runs(experiment_ids=["1"], order_by=["start_time DESC"], max_results=1)

run = runs[0]
run_id = run.info.run_id

print("Run ID:", run_id)
print("Artifact URI:", run.info.artifact_uri)

artifacts = client.list_artifacts(run_id)
print("Artifacts:", artifacts)

Run ID: ea51104761104563a51e5064b340cb1b
Artifact URI: /mlflow_data/artifacts/1/ea51104761104563a51e5064b340cb1b/artifacts
Artifacts: [<FileInfo: file_size=None, is_dir=True, path='model'>]
